In [1]:
import json
import os
import pandas as pd
import itertools
from augur.utils import json_to_tree
from scipy.stats import linregress
import matplotlib.pyplot as plt
from collections import Counter
from Bio import SeqIO
from Bio.Seq import Seq

In [2]:
def readin_virus_config(virus):
    """
    Read in the config file for this virus to get the paths to alignment, metadata files, etc 
    as well as metadata about the virus such as how many subtypes, 
    and which genes are receptor-binding or polymerase
    """
    config_json = f'config/adaptive_evo_config_{virus}.json'
    with open(config_json) as json_handle:
        configs = json.load(json_handle)
        
    return configs

In [3]:
def map_nuc_to_gene(virus):
    """
    Create a dictionary mapping each nucleotide position in the genome to what gene it is in
    
    This is set to do full HA as of now,
    to get sigpep, HA1, HA2 separately, will need to change 'actual_CDS' to 'CDS' 
    and include those genes in 'all_genes'
    """
    configs = readin_virus_config(virus)
    
    # path to reference file
    reference_file = configs['reference_file']
    # get all genes
    all_genes = ['ha']
    
    # get coordinates for all genes
    # store nucleotide coordinates as value and gene name as key
    gene_coordinates = {}
    
    
    # some reference files are genbank, some are fasta and gff
    if reference_file[-3:]=='.gb':
        for seq_record in SeqIO.parse(reference_file, "genbank"):
            for feature in seq_record.features:
                if feature.type == 'actual_CDS':
                    if 'gene' in feature.qualifiers.keys():
                        gene_name = feature.qualifiers['gene'][0].lower()
                        if gene_name in all_genes:

                            gene_location = [int(feature.location.start)+3, int(feature.location.end)]
                            gene_coordinates[gene_name] = gene_location



    # rather than storing names of HA subunits separately, call them all HA
#     ha_subunits = ['sigpep', 'ha1', 'ha2']
#     if 'ha1' in gene_coordinates.keys():
#         gene_coordinates = {'ha': [gene_coordinates['sigpep'][0], gene_coordinates['ha2'][1]]}
    
    # map each nt position in a coding region to what gene it is in
    nt_to_gene_mapper = {}
    
    for g, p in gene_coordinates.items():
        # 1-based coordinates
        for x in range(p[0]+1, p[1]+1):
            nt_to_gene_mapper[x] = g
            
                    
    return nt_to_gene_mapper

In [4]:
def readin_tree(virus):
    """
    Read in the nextstrain json tree
    """
    configs = readin_virus_config(virus)
    path_to_tree = configs['tree_file']

    #read in the tree
    with open(path_to_tree, 'r') as f:
        tree_json = json.load(f)
        
    # convert json to Bio.Phylo
    tree = json_to_tree(tree_json)
    
    return tree

In [5]:
def curate_muts(mut_dict):
    """
    remove indels and ambiguous sequencing from muts 
    """
    
    curated_muts_dict = {}
    
    for g, ms in mut_dict.items():
        # exclude all indels and ambiguous (X AA or N nt)
        curated_ms = [m for m in ms if '-' not in m]
        if g == 'nuc':
            curated_ms = [m for m in curated_ms if 'N' not in m]
        else:
            curated_ms = [m for m in curated_ms if 'X' not in m]

        curated_muts_dict[g] = curated_ms
            
    
    return curated_muts_dict

In [6]:
def get_gene_for_nuc_muts(list_of_muts, nt_to_gene_mapper):
    """
    find what gene each nuc mut occurs in
    return a dict with gene name and list of nuc muts
    """
    
    nuc_muts_per_gene = {}
    
    for m in list_of_muts:
        nt_pos = int(m[1:-1])
        if nt_pos in nt_to_gene_mapper:
            g = nt_to_gene_mapper[nt_pos]
        else:
            g = 'noncoding'
        if g in nuc_muts_per_gene:
            nuc_muts_per_gene[g].append(m)
        else:
            nuc_muts_per_gene[g] = [m]
            
    return nuc_muts_per_gene

In [7]:
def get_muts_from_root(virus):
    """
    for each tip, just find the number of muts from root 
    (this will be muts from reference where reference sequence is the root)
    this is instead of calculating them relative to the last internal branch
    separate them by gene and get "syn" as nuc_muts minus aa_muts
    """
    
    tree = readin_tree(virus)
    
    nt_to_gene_mapper = map_nuc_to_gene(virus)
    
    # get all genes
    all_genes = ['ha']
    
    # save syn, nonsyn mut counts and date for each tip
    all_node_info = {}
    
    # find muts relative to root for each tip
    for node in tree.find_clades(terminal=True):
        date = node.node_attrs['num_date']['value']
        
        path_to_root = tree.get_path(node)
        
        # amass all muts from root to tip
        mut_counts_this_tip = {}
        
        # count syn and nonsyn muts per gene on each branch back to the root
        for p in path_to_root:
            p_muts = p.branch_attrs['mutations']
            # remove indels, and ambiguous sequencing
            curated_muts = curate_muts(p_muts)
            # limit AA muts to just those in the specified genes 
            # need to combine sigpep, ha1 and ha2 into all HA 
            curated_muts = {k.lower():v for k,v in curated_muts.items() if k.lower() in ['sigpep', 'ha1', 'ha2'] or k=='nuc'}

#             curated_muts = {k.lower():v for k,v in curated_muts.items() if k.lower() in all_genes or k=='nuc'}

            # get number of syn and nonsyn for each gene
            # if there are muts
            if 'nuc' in curated_muts:

                nuc_muts_by_gene = get_gene_for_nuc_muts(curated_muts['nuc'], nt_to_gene_mapper)

                for g, nuc_ms in nuc_muts_by_gene.items():
                    if g!= 'noncoding':
                        # for influenza HA, AA muts are listed as sigpep, ha1, ha2. want to combine all
                        if g=='ha':
                            nonsyn_muts = len(curated_muts.get('sigpep', [])) + len(curated_muts.get('ha1', []))+ len(curated_muts.get('ha2', []))
                        elif g in curated_muts:
                            nonsyn_muts = len(curated_muts[g])
                        else:
                            nonsyn_muts = 0

                        syn_muts = len(nuc_ms) - nonsyn_muts
                        if g in mut_counts_this_tip:
                            mut_counts_this_tip[g]['syn']+=syn_muts
                            mut_counts_this_tip[g]['nonsyn']+=nonsyn_muts
                        else:
                            mut_counts_this_tip[g] = {'syn': syn_muts, 'nonsyn': nonsyn_muts}
                
        all_node_info[node.name] = {'date': date, 'mut_counts': mut_counts_this_tip}
            
            
    return all_node_info

In [8]:
def get_rates(virus):
    """
    get rate of syn and nonsyn muts (counting tip muts from root seq as reference)
    """
    
    all_node_info = get_muts_from_root(virus)
    
    all_genes = ['ha']
    
    # get length of each gene, to normalize rates
    nt_to_gene_mapper = map_nuc_to_gene(virus)
    gene_lens = Counter(nt_to_gene_mapper.values())
    
    results_by_gene = {}
    
    # get X and Y data for each gene
    for gene in all_genes:
        dates = []
        syn_counts = []
        nonsyn_counts = []
        
        for accession, info in all_node_info.items():
            if gene in info['mut_counts']:
                dates.append(info['date'])
                counts = info['mut_counts'][gene]
                syn_counts.append(counts['syn'])
                nonsyn_counts.append(counts['nonsyn'])
        
        # if there are any entries for this gene, compute slope 
        # (for flu viruses, this is done by segment so there won't be entries for every gene)
        if len(dates)>2:
            # now find slope via linear regression        

            lr_syn = linregress(dates, syn_counts)
            lr_nonsyn = linregress(dates, nonsyn_counts)
            results_by_gene[gene] = {
                'syn_rate': lr_syn.slope,
                'syn_rate_per_nt': lr_syn.slope/gene_lens[gene],
                'syn_intercept': lr_syn.intercept,
                'nonsyn_rate': lr_nonsyn.slope,
                'nonsyn_rate_per_nt': lr_nonsyn.slope/gene_lens[gene],
                'nonsyn_intercept': lr_nonsyn.intercept,
                'syn_r2': lr_syn.rvalue**2,
                'nonsyn_r2': lr_nonsyn.rvalue**2,
            }
            
#         if gene=='h':
#             fig, ax = plt.subplots()
#             ax = plt.scatter(dates, nonsyn_counts)
        
    return results_by_gene

In [9]:
def save_results(viruses):
    """
    Compute rates for only the viruses not yet present in out_path, and append.
    If out_path doesn't exist, compute for all and create it.
    """
    # Ensure the output directory exists
    out_path = "rates_results/avian_rates_from_ref.tsv"
    out_dir = os.path.dirname(out_path)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir)

    # Load existing results if file exists
    if os.path.exists(out_path):
        existing_df = pd.read_csv(out_path, sep=",|\\t", engine="python")
        done = set(existing_df["virus"].unique()) if not existing_df.empty else set()
    else:
        existing_df = None
        done = set()

    # Filter only viruses not yet processed
    to_run = [v for v in viruses if v not in done]
    if not to_run:
        return existing_df if existing_df is not None else pd.DataFrame()
    all_virus_rates = []

    for virus in to_run:

        results_by_gene = get_rates(virus)
        for g, d in results_by_gene.items():
            all_virus_rates.append({
                'virus':  virus, 'gene': g, 'mut_type': 'syn',
                'rate': d['syn_rate'], 'rate_per_nt': d['syn_rate_per_nt'], 'r2': d['syn_r2']
            })
            all_virus_rates.append({
                'virus':  virus, 'gene': g, 'mut_type': 'nonsyn',
                'rate': d['nonsyn_rate'], 'rate_per_nt': d['nonsyn_rate_per_nt'], 'r2': d['nonsyn_r2']
            })


    new_df = pd.DataFrame(all_virus_rates)

    # Combine with existing results if present
    if existing_df is not None and not existing_df.empty:
        combined = pd.concat([existing_df, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=['virus', 'gene', 'mut_type'])
    else:
        combined = new_df

    combined.to_csv(out_path, index=False, sep="\t")


In [41]:
save_results(['avianH1a', 'avianH1b', 'avianH3a', 'avianH3b', 'avianH3c'])